In [1]:
import argparse
import os
import numpy as np
from Bio import Phylo
import sys
import os
import copy
import math
import json
import pandas as pd

# Add parent directory to sys.path
sys.path.append(os.path.abspath(".."))
from simulation_utils import *

def smooth_scores(scores, window=5, poly=2):
    if len(scores) < window:
        return scores
    return savgol_filter(scores, window_length=window, polyorder=poly)

def parsimony_vs_lambda(out, parasites, hosts, host_W_matrices, par_W_matrices, flip_cost_matrix, steps, lower, upper):
    """
    Compute total parsimony for a range of λ values between 0 and 1.
    Returns arrays of lambdas and parsimony scores.
    """
    # filter lambda from 0.5 to 0.9
    lambdas = np.linspace(lower, upper, steps)
    scores = []

    for lam in lambdas:
        cut_result = solve_network_cut(
            out, host_W_matrices=host_W_matrices, par_W_matrices=par_W_matrices,
            flip_cost_matrix=flip_cost_matrix,
            lambda_param=lam
        )
        mat = cut_result["new_matrix"]

        # total parsimony across parasite×host
        total = 0
        for p in parasites:
            leaf_states = {h: cut_result["new_cell_state"][(p,h)] for h in hosts}
            total += sankoff(out["host_trees"][0], leaf_states, np.array([[0,1],[1,0]]))
        for h in hosts:
            leaf_states = {p: cut_result["new_cell_state"][(p,h)] for p in parasites}
            total += sankoff(out["par_trees"][0], leaf_states, np.array([[0,1],[1,0]]))

        scores.append(total)

    return lambdas, scores

def find_elbow_parsimony(out, parasites, hosts, host_W_matrices, par_W_matrices, flip_cost_matrix, lower, upper, outdir="experiments"):
    # Run your sweep
    lambdas, scores = parsimony_vs_lambda(out, parasites, hosts, host_W_matrices, par_W_matrices, flip_cost_matrix, steps=50, lower=lower, upper=upper)

    # Use KneeLocator to detect elbow
    # Smooth scores to reduce noise
    # scores = smooth_scores(scores, window=5, poly=1)
        # Smooth to reduce zig-zag

    lambdas = np.array(lambdas)
    scores = np.array(scores)

    # Smooth to reduce zig-zag
    smooth = smooth_scores(scores, window=3, poly=2)
    smooth_lambdas = lambdas[:len(smooth)]

    # Compute slope
    diffs = np.diff(smooth) / np.diff(smooth_lambdas)

    # Step 1: find steepest slope
    idx_steep = np.argmin(diffs)  # most negative slope
    cutoff_lambda = smooth_lambdas[idx_steep]

    # Step 2: run KneeLocator only on the part after cutoff
    mask_after = lambdas >= cutoff_lambda
    lambdas_after = lambdas[mask_after]
    scores_after = scores[mask_after]

    if len(lambdas_after) < 3:  # fallback if too short
        elbow_lambda = cutoff_lambda
    else:
        kneedle = KneeLocator(
            lambdas_after, scores_after,
            curve="convex", direction="decreasing"
        )
        elbow_lambda = kneedle.elbow if kneedle.elbow else cutoff_lambda

    print(f"[Elbow detection] λ ≈ {elbow_lambda:.3f} (cutoff={cutoff_lambda:.3f})")

    # Plot
    plt.figure(figsize=(7,5))
    plt.plot(lambdas, scores, "bo-", label="Parsimony score")
    plt.axvline(cutoff_lambda, color="orange", linestyle="--",
                label=f"Steep cutoff λ={cutoff_lambda:.3f}")
    if elbow_lambda is not None:
        plt.axvline(elbow_lambda, color="red", linestyle="--",
                    label=f"Elbow λ={elbow_lambda:.3f}")
    plt.xlabel("λ")
    plt.ylabel("Parsimony")
    plt.title("Parsimony vs λ (Steepness + Knee)")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(outdir, "parsimony_vs_lambda_elbow.png"))
    plt.close()

    return elbow_lambda, lambdas, scores


def read_interaction_matrix(csv_file, host_tree_file, virus_tree_file):
    """
    Read host-virus interaction matrix CSV and reorder according to tree leaves.
    Treat viruses as parasites.
    
    Returns:
        cell_state: dict with keys (virus, host) -> 0/1
        host_tree: Bio.Phylo tree object
        virus_tree: Bio.Phylo tree object
    """

    # Load trees
    host_tree = Phylo.read(host_tree_file, "newick")
    virus_tree = Phylo.read(virus_tree_file, "newick")

    # Get leaf order
    host_order = [t.name for t in host_tree.get_terminals()]
    virus_order = [t.name for t in virus_tree.get_terminals()]

    print(f"Number of hosts in tree: {len(host_order)}")
    print(f"Number of viruses in tree: {len(virus_order)}")

    # Load interaction matrix CSV
    df = pd.read_csv(csv_file, index_col=0)  # assuming rows=viruses, columns=hosts
    # Ensure names match the expected tree leaf names
    df = df.loc[df.index.intersection(virus_order), df.columns.intersection(host_order)]

    # Reorder according to tree leaves
    df = df.reindex(index=virus_order, columns=host_order, fill_value=0)

    # Build cell_state dictionary
    cell_state = {}
    for virus in virus_order:
        for host in host_order:
            cell_state[(virus, host)] = int(df.at[virus, host])

    return cell_state, host_tree, virus_tree






def matrix_builder(out, r01_h, r10_h, r01_p, r10_p):
    # Build weight matrices for each parasite’s host tree
    host_W_matrices = []
    for t in out["host_trees"]:
        W, nodes = build_weight_matrix(t, r01_h, r10_h)
        host_W_matrices.append((W, nodes))

    # Build weight matrices for each host’s parasite tree
    par_W_matrices = []
    for t in out["par_trees"]:
        W, nodes = build_weight_matrix(t, r01_p, r10_p)
        par_W_matrices.append((W, nodes))

    # Flip cost matrix
    flip_cost_matrix = build_flip_cost_matrix(len(out["par_leaves"]), len(out["host_leaves"]), cost=1.0)
    return host_W_matrices, par_W_matrices, flip_cost_matrix

# def run_multiple_seeds(out, parasites, hosts, host_W_matrices, par_W_matrices, flip_cost_matrix,
#                        seeds, corrupt=5, outdir="experiments"):
#     """
#     Run the network recovery pipeline for multiple seeds with given corruption,
#     and return the result with the best F1 score.
#     """
#     best_metrics = {"f1": -1}
#     best_result = None
#     out_original = out.copy()  # Keep original for each run

#     for seed in seeds:
#         print(f"\n=== Running seed {seed} ===")
#         out = out_original.copy()  # Reset to original each time

#         mat, parasites, hosts = get_interaction_matrix(out)
#         random_flips = min(corrupt, np.sum(mat) - 1)
#         corrupt_mat, hidden = corrupt_matrix(mat, num_flips=random_flips, seed=seed)

#         # Save hidden cells for metrics
#         hidden_cells = [(parasites[i], hosts[j]) for i, j in hidden]

#         highlight_corrupted = {"corrupted": hidden_cells}
#         plot_matrix(corrupt_mat, parasites, hosts,
#                     filename=os.path.join(outdir, "corrupted.png"),
#                     highlight=highlight_corrupted)

#         # Update out["cell_state"] with corrupted values
#         corrupt_cell_state = {}
#         for i, p in enumerate(parasites):
#             for j, h in enumerate(hosts):
#                 corrupt_cell_state[(p, h)] = int(corrupt_mat[i, j])
#         out["cell_state"] = corrupt_cell_state
        

#         lambda_param, cut_result = binary_search_lambda(
#             out, parasites, hosts, hidden_cells=hidden_cells,
#             target_flips=len(hidden_cells),
#             host_W_matrices=host_W_matrices, par_W_matrices=par_W_matrices,
#             flip_cost_matrix=flip_cost_matrix,
#             tol=0, max_iter=20
#         )
#         flips = cut_result["flips"]

#         # Compute metrics
#         metrics = compute_metrics(hidden_cells, flips)
#         print(f"Metrics for seed {seed}: {metrics}")

#         # Update best if F1 improved
#         if metrics["f1"] > best_metrics.get("f1", -1):
#             best_metrics = metrics
#             best_result = {
#                 "seed": seed,
#                 "cut_result": cut_result,
#                 "metrics": metrics,
#                 "parasites": parasites,
#                 "hosts": hosts
#             }

#     # Visualize best result
#     if best_result:
#         print(f"\nBest seed: {best_result['seed']} with metrics: {best_result['metrics']}")
#         plot_matrix(
#             best_result["cut_result"]["new_matrix"],
#             best_result["parasites"],
#             best_result["hosts"],
#             filename=os.path.join(outdir, "best_flipped_matrix.png"),
#             highlight={"flipped": [(p, h) for p, h, old, new in best_result["cut_result"]["flips"]]}
#         )

#     return best_result





In [27]:
outdir = "experiments"
association_csv = "interaction_matrix_tree_leaves.csv"
host_tree = "virion_bat_cov_output/trees/hosts_tree.newick"
virus_tree = "virion_bat_cov_output/trees/viruses_tree.newick"


os.makedirs(outdir, exist_ok=True)

# Read associations + trees
cell_state, base_host_tree, base_par_tree = read_interaction_matrix(
    association_csv, host_tree, virus_tree
)

base_host_tree = rescale_tree(base_host_tree)
base_par_tree = rescale_tree(base_par_tree)

host_leaves = [t.name for t in base_host_tree.get_terminals()]
par_leaves = [t.name for t in base_par_tree.get_terminals()]

out = {
    "host_trees": [copy.deepcopy(base_host_tree) for _ in range(len(par_leaves))],
    "par_trees": [copy.deepcopy(base_par_tree) for _ in range(len(host_leaves))],
    "cell_state": cell_state,
    "host_leaves": host_leaves,
    "par_leaves": par_leaves
}

# Build the matrix
mat, parasites, hosts = get_interaction_matrix(out)

# Save the matrix visualization
plot_matrix(mat, parasites, hosts, filename=os.path.join(outdir, "original_matrix.png"))







Number of hosts in tree: 233
Number of viruses in tree: 94
Matrix saved as experiments/original_matrix.png


In [28]:
out_original = out.copy()  # Keep original for each run

In [42]:
out = out_original.copy()  # Reset to original each time
corrupt = 200
seed = 42
r01_p, r10_p, r01_h, r10_h = 0.5, 0.5, 0.5, 0.5
lambda_param = 0.95
host_W_matrices, par_W_matrices, flip_cost_matrix = matrix_builder(out, r01_h, r10_h, r01_p, r10_p)

In [41]:
# ---------------------------
# Step 1: Create corrupted matrix (random 1→0 flips)
# ---------------------------
#     best_result = run_multiple_seeds(
#     out, parasites, hosts,
#     host_W_matrices, par_W_matrices, flip_cost_matrix,
#     seeds=range(100),  # example seeds
#     corrupt=corrupt,
#     outdir=args.outdir
# )

if corrupt > 0:
    random_flips = min(corrupt, np.sum(mat) - 1)
    corrupt_mat, hidden = corrupt_matrix(mat, num_flips=random_flips, seed=seed)

    # Save hidden cells for metrics
    hidden_cells = [(parasites[i], hosts[j]) for i, j in hidden]

    highlight_corrupted = {"corrupted": hidden_cells}
    plot_matrix(corrupt_mat, parasites, hosts,
                filename=os.path.join(outdir, "corrupted.png"),
                highlight=highlight_corrupted)

    # Update out["cell_state"] with corrupted values
    corrupt_cell_state = {}
    for i, p in enumerate(parasites):
        for j, h in enumerate(hosts):
            corrupt_cell_state[(p, h)] = int(corrupt_mat[i, j])
    out["cell_state"] = corrupt_cell_state
    

    lambda_param, cut_result = binary_search_lambda(
        out, parasites, hosts, hidden_cells=hidden_cells,
        target_flips=len(hidden_cells),
        host_W_matrices=host_W_matrices, par_W_matrices=par_W_matrices,
        flip_cost_matrix=flip_cost_matrix,
        tol=0, max_iter=20
    )
    # cut_result = solve_network_cut(
    #     out, host_W_matrices=host_W_matrices, par_W_matrices=par_W_matrices,
    #     flip_cost_matrix=flip_cost_matrix,
    #     lambda_param=args.lambda_param)
else:
    hidden_cells = []
    cut_result = solve_network_cut(
        out, host_W_matrices=host_W_matrices, par_W_matrices=par_W_matrices,
        flip_cost_matrix=flip_cost_matrix,
        lambda_param=lambda_param)
    
# ---------------------------
# Step 2: Run network cut recovery on corrupted input
# ---------------------------


new_mat = cut_result["new_matrix"]
new_cell_state = cut_result["new_cell_state"]
flips = cut_result["flips"]

highlight_flipped = {"flipped": [(p, h) for p, h, old, new in flips]}
print(f"Number of flips performed: {len(flips)}")
for p, h, old, new in flips:
    print(f"Cell ({p},{h}): {old} -> {new}")

# ---------------------------
# Step 3: Save recovered matrix with algorithm flips
# ---------------------------
plot_matrix(new_mat, parasites, hosts,
            filename=os.path.join(outdir, "flipped_matrix.png"),
            highlight=highlight_flipped)






# inside main(), after you have flips and hidden_cells
metrics = compute_metrics(hidden_cells, flips)
print("Metrics:", metrics)






Matrix saved as experiments/corrupted.png
[iter 0] λ=0.5000 → flips=1
[iter 1] λ=0.7500 → flips=8
[iter 2] λ=0.8750 → flips=12
[iter 3] λ=0.9375 → flips=25
[iter 4] λ=0.9688 → flips=2644
[iter 5] λ=0.9531 → flips=166
[iter 6] λ=0.9609 → flips=2490
[iter 7] λ=0.9570 → flips=167
[iter 8] λ=0.9590 → flips=167
[iter 9] λ=0.9600 → flips=167
[iter 10] λ=0.9604 → flips=167
[iter 11] λ=0.9607 → flips=2490
[iter 12] λ=0.9606 → flips=167
[iter 13] λ=0.9606 → flips=2490
[iter 14] λ=0.9606 → flips=2490
[iter 15] λ=0.9606 → flips=167
[iter 16] λ=0.9606 → flips=2490
[iter 17] λ=0.9606 → flips=2490
[iter 18] λ=0.9606 → flips=167
[iter 19] λ=0.9606 → flips=2490

Best λ=0.9570 → flips=167, correct=18/200
Number of flips performed: 167
Cell (virus_3433779,host_9433): 0 -> 1
Cell (virus_694001,host_187007): 0 -> 1
Cell (virus_694001,host_9433): 0 -> 1
Cell (virus_694001,host_187008): 0 -> 1
Cell (virus_1805463,host_159858): 0 -> 1
Cell (virus_1805463,host_376555): 0 -> 1
Cell (virus_1805463,host_169756):

In [16]:
cut_result = solve_network_cut(
        out, host_W_matrices=host_W_matrices, par_W_matrices=par_W_matrices,
        flip_cost_matrix=flip_cost_matrix,
        lambda_param=0.9032)
new_mat = cut_result["new_matrix"]
new_cell_state = cut_result["new_cell_state"]
flips = cut_result["flips"]

highlight_flipped = {"flipped": [(p, h) for p, h, old, new in flips]}
print(f"Number of flips performed: {len(flips)}")
for p, h, old, new in flips:
    print(f"Cell ({p},{h}): {old} -> {new}")

# ---------------------------
# Step 3: Save recovered matrix with algorithm flips
# ---------------------------
plot_matrix(new_mat, parasites, hosts,
            filename=os.path.join(outdir, "flipped_matrix.png"),
            highlight=highlight_flipped)






# inside main(), after you have flips and hidden_cells
metrics = compute_metrics(hidden_cells, flips)
print("Metrics:", metrics)

Number of flips performed: 193
Cell (virus_2691588,host_187007): 0 -> 1
Cell (virus_2691588,host_187008): 0 -> 1
Cell (virus_2691588,host_438766): 0 -> 1
Cell (virus_2691588,host_258934): 0 -> 1
Cell (virus_3433780,host_9433): 0 -> 1
Cell (virus_3433780,host_187008): 0 -> 1
Cell (virus_3433780,host_438766): 0 -> 1
Cell (virus_3433780,host_258934): 0 -> 1
Cell (virus_694000,host_187007): 0 -> 1
Cell (virus_694000,host_187008): 0 -> 1
Cell (virus_3119327,host_187007): 0 -> 1
Cell (virus_3119327,host_9433): 0 -> 1
Cell (virus_3119327,host_187008): 0 -> 1
Cell (virus_3119327,host_438766): 0 -> 1
Cell (virus_3119327,host_258934): 0 -> 1
Cell (virus_2809631,host_187007): 0 -> 1
Cell (virus_2809631,host_9433): 0 -> 1
Cell (virus_2809631,host_187008): 0 -> 1
Cell (virus_2809631,host_438766): 0 -> 1
Cell (virus_2809631,host_258934): 0 -> 1
Cell (virus_3433779,host_9433): 0 -> 1
Cell (virus_694001,host_187007): 0 -> 1
Cell (virus_694001,host_187008): 0 -> 1
Cell (virus_1335626,host_59485): 0 -> 

In [43]:
elbow_lambda, lambdas, scores = find_elbow_parsimony(
    out, parasites, hosts, host_W_matrices, par_W_matrices, flip_cost_matrix, lower=0, upper=1, outdir=outdir
)

cut_result_elbow = solve_network_cut(
    out, host_W_matrices=host_W_matrices, par_W_matrices=par_W_matrices,
    flip_cost_matrix=flip_cost_matrix,
    lambda_param=elbow_lambda
)
flips_elbow = cut_result_elbow["flips"]

highlight_flipped = {"flipped": [(p, h) for p, h, old, new in flips_elbow]}
print(f"Number of flips performed (elbow): {len(flips_elbow)}")
# for p, h, old, new in flips_elbow:
#     print(f"Cell ({p},{h}): {old} -> {new}")
plot_matrix(cut_result_elbow["new_matrix"], parasites, hosts,
            filename=os.path.join(outdir, "flipped_matrix_elbow.png"),
            highlight=highlight_flipped)

metrics_elbow = compute_metrics(hidden_cells, flips_elbow)
print("Elbow Metrics:", metrics_elbow)

[Elbow detection] λ ≈ 0.959 (cutoff=0.959)
Number of flips performed (elbow): 2606
Matrix saved as experiments/flipped_matrix_elbow.png
Elbow Metrics: {'precision': 0.0, 'recall': 0.0, 'f1': 0, 'TP': 0, 'FP': 2606, 'FN': 200}
